# P5-7 · GĐ 7 · Đóng gói sản phẩm

Toàn bộ code của bài, chạy từ trên xuống như trên trang bài học. Kết quả đã chạy sẵn được giữ trong notebook; bấm **Runtime → Run all** để chạy lại. Ô có dòng đầu `# Cần GPU` thì đổi **Runtime → Change runtime type → T4 GPU**; ô cần API key thì đặt khóa trong Colab Secrets.

In [ ]:
!pip install -q diffusers gradio

## Lý thuyết

In [ ]:
# Cần GPU và pipe đã nạp ở P5.4: chạy trên Colab (launch mở giao diện web)
import gradio as gr

def generate(prompt, steps, seed):
    # gọi pipeline đã nạp sẵn ở cấp module; trả về ảnh PIL
    return pipe(prompt, num_inference_steps=int(steps),
                generator=torch.Generator("cuda").manual_seed(int(seed))).images[0]

demo = gr.Interface(fn=generate,
                    inputs=[gr.Textbox(label="Mô tả"), gr.Slider(10, 40, value=25, step=1, label="Số bước"),
                            gr.Number(value=0, label="Seed")],
                    outputs=gr.Image(label="Ảnh"))
demo.queue(max_size=20).launch()

## Bài tập 5: App Gradio {code}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
# Cần GPU, mạng và LoRA đã train: chạy trên Colab
import gradio as gr, torch
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler

pipe = StableDiffusionPipeline.from_pretrained("stable-diffusion-v1-5/stable-diffusion-v1-5", torch_dtype=torch.float16).to("cuda")
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
pipe.load_lora_weights("lora-tthstyle")

def generate(prompt, negative, steps, guidance, lora_scale, seed):
    if not prompt.strip():
        raise gr.Error("Hãy nhập mô tả ảnh.")
    g = torch.Generator("cuda").manual_seed(int(seed))
    return pipe(prompt[:300], negative_prompt=negative[:300], num_inference_steps=int(steps), guidance_scale=guidance,
                generator=g, cross_attention_kwargs={"scale": lora_scale}).images[0]

with gr.Blocks(title="Tạo ảnh sản phẩm nội thất") as demo:
    gr.Markdown("## Tạo ảnh sản phẩm nội thất\nStable Diffusion 1.5 + LoRA phong cách ảnh sản phẩm.")
    with gr.Row():
        with gr.Column():
            prompt = gr.Textbox(label="Mô tả (tiếng Anh)", lines=2)
            negative = gr.Textbox(label="Không muốn có", value="blurry, text, watermark")
            steps = gr.Slider(10, 40, value=25, step=1, label="Số bước")
            guidance = gr.Slider(1, 12, value=7, step=0.5, label="Guidance")
            lora_scale = gr.Slider(0, 1, value=0.8, step=0.05, label="Độ mạnh phong cách")
            seed = gr.Number(value=42, precision=0, label="Seed")
            btn = gr.Button("Tạo ảnh", variant="primary")
        out = gr.Image(label="Kết quả")
    gr.Examples([["a vanity table with round mirror, tthstyle product photo"],
                 ["a rattan armchair, tthstyle product photo"],
                 ["a ceramic table lamp, tthstyle product photo"]], inputs=[prompt])
    btn.click(generate, [prompt, negative, steps, guidance, lora_scale, seed], out)

demo.queue(max_size=20).launch()

## Bài tập 6: Mô phỏng hàng đợi GPU giờ cao điểm {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
toc_do = 5 / 60                                      # yêu cầu mỗi giây
den = np.cumsum(rng.exponential(1 / toc_do, 2000))
den = den[den < 2 * 3600]                            # 2 giờ
PHUC_VU = 10.0                                       # giây mỗi ảnh

def mo_phong(so_gpu):
    # TODO: trả về mảng thời gian chờ (giây) của từng yêu cầu
    return np.zeros(len(den))

In [ ]:
def mo_phong(so_gpu):
    ranh = np.zeros(so_gpu); cho = []
    for t in den:
        g = ranh.argmin()
        bat_dau = max(t, ranh[g])
        cho.append(bat_dau - t)
        ranh[g] = bat_dau + PHUC_VU
    return np.array(cho)

cho = {k: mo_phong(k) for k in (1, 2)}
for k, c in cho.items():
    print(f"{k} GPU: tải {toc_do * PHUC_VU / k:.0%}, chờ TB {c.mean():.1f} s, chờ > 30 s: {(c > 30).mean():.1%}, lâu nhất {c.max():.0f} s")

1 GPU: tải 83%, chờ TB 14.5 s, chờ > 30 s: 17.6%, lâu nhất 80 s
2 GPU: tải 42%, chờ TB 0.7 s, chờ > 30 s: 0.0%, lâu nhất 13 s


## Bài tập 7: Kiểm tra tham số cho API tạo ảnh {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
CHAN = {"bạo lực", "khỏa thân"}
mau = [{"prompt": "bàn trang điểm gỗ sồi, ánh sáng tự nhiên", "w": 512, "h": 512, "steps": 20, "n": 2},
       {"prompt": "", "w": 512, "h": 512, "steps": 20, "n": 1},
       {"prompt": "phòng ngủ tối giản", "w": 500, "h": 512, "steps": 20, "n": 1},
       {"prompt": "chân dung", "w": 1024, "h": 1024, "steps": 50, "n": 4},
       {"prompt": "cảnh bạo lực trên phố", "w": 512, "h": 512, "steps": 20, "n": 1},
       {"prompt": "ghế đôn nhung", "w": 768, "h": 512, "steps": 30, "n": 1}]

def kiem_tra(req):
    # TODO
    return []

In [ ]:
def kiem_tra(req):
    loi, p = [], req["prompt"].strip()
    if not 1 <= len(p) <= 300:
        loi.append("prompt phải có từ 1 đến 300 ký tự")
    if any(w in p.lower() for w in CHAN):
        loi.append("prompt chứa nội dung không được phép")
    for k in ("w", "h"):
        if req[k] % 8 or not 256 <= req[k] <= 1024:
            loi.append(f"{k} phải là bội số của 8, từ 256 đến 1024 (nhận {req[k]})")
    if not 1 <= req["steps"] <= 50:
        loi.append("steps phải từ 1 đến 50")
    if not 1 <= req["n"] <= 4:
        loi.append("n phải từ 1 đến 4")
    chi_phi = req["w"] * req["h"] * req["steps"] * req["n"] / 512 ** 2 / 20
    if chi_phi > 8:
        loi.append(f"yêu cầu quá nặng ({chi_phi:.0f} đơn vị, tối đa 8): giảm kích thước, số bước hoặc số ảnh")
    return loi

for r in mau:
    print(r["prompt"][:25] or "(rỗng)", "→", kiem_tra(r) or "hợp lệ")

bàn trang điểm gỗ sồi, án → hợp lệ
(rỗng) → ['prompt phải có từ 1 đến 300 ký tự']
phòng ngủ tối giản → ['w phải là bội số của 8, từ 256 đến 1024 (nhận 500)']
chân dung → ['yêu cầu quá nặng (40 đơn vị, tối đa 8): giảm kích thước, số bước hoặc số ảnh']
cảnh bạo lực trên phố → ['prompt chứa nội dung không được phép']
ghế đôn nhung → hợp lệ


## Bài tập 8: Cache ảnh theo prompt và seed {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import hashlib, json
log = [("Bàn trang điểm gỗ sồi", 42), ("bàn trang điểm gỗ sồi ", 42), ("Gương LED tròn", 7), ("bàn  trang điểm gỗ sồi", 42),
       ("Gương LED tròn", 8), ("Ghế đôn nhung", 1), ("gương led tròn", 7), ("Ghế đôn nhung", 1),
       ("Bàn trang điểm gỗ sồi", 43), ("Kệ mỹ phẩm", 5), ("kệ mỹ phẩm", 5), ("Ghế đôn nhung", 2)]
CO_DINH = {"w": 512, "h": 512, "steps": 20, "model": "sd15-lora-shop-v3"}

def khoa_cache(prompt, seed):
    # TODO
    return ""

In [ ]:
def khoa_cache(prompt, seed):
    goc = {"prompt": " ".join(prompt.lower().split()), "seed": seed, **CO_DINH}
    return hashlib.sha256(json.dumps(goc, sort_keys=True, ensure_ascii=False).encode()).hexdigest()

cache, trung = {}, 0
for prompt, seed in log:
    k = khoa_cache(prompt, seed)
    if k in cache:
        trung += 1
    else:
        cache[k] = f"anh_{len(cache)}.png"          # thật: chạy model rồi lưu ảnh
print(f"trúng cache {trung}/{len(log)}, sinh mới {len(cache)} ảnh, tiết kiệm {trung * 40} đồng")

trúng cache 5/12, sinh mới 7 ảnh, tiết kiệm 200 đồng
